# SWE-bench Verified: PathPack на реальных задачах

Единый исследовательский блокнот для этапа 2: подготовка выборки, фиксация репозиториев на `base_commit`, графы и сравнение BM25 / Graph Expansion / PathPack. В корне проекта активируйте `.venv`, установленную через `pip install -e '.[notebook]'`, и откройте этот файл в Jupyter. Docker и запуск исправляющих тесты агентов здесь не нужны.

**Протокол:** 80 задач выбираются фиксированным seed 42 и делятся на 20 dev / 60 test. Текст issue — единственный запрос. Gold patch хранится отдельно и используется только в ячейках оценки. Основные сравнения используют одинаковые задачи и бюджеты. Гиперпараметры PathPack подбираются на dev; test остаётся отложенным.

**Ограничение v0.2:** `tokens_est` — оценка `ceil(UTF-8 bytes / 4)`, а не токенизация Gemma. До выбора целевой модели результаты корректно трактовать как сравнение при одинаковом proxy-бюджете, не как точный лимит модельного контекста. Метрики локализации не означают успешное исправление бага.

In [4]:
from pathlib import Path
import sys, os, json, re, time, subprocess, math
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Откройте блокнот из корня CodeGraph-RAG'
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'output' / 'benchmark' / 'hf_cache'))
if list((ROOT / 'output' / 'benchmark' / 'hf_cache').glob('SWE-bench___swe-bench_verified/default/0.0.0/*')): os.environ.setdefault('HF_HUB_OFFLINE', '1')
from datasets import load_dataset
from codegraph_rag.benchmark import build_manifests, read_jsonl
from codegraph_rag.parser import scan_repository
from codegraph_rag.retrieval import make_chunks, select_context
BUDGETS = (1024, 2048, 4096)
OUT = ROOT / 'output' / 'benchmark'
ISSUES = OUT / 'issues.jsonl'
GOLD = OUT / 'gold_labels.jsonl'  # открывать только в scoring-ячейках
SNAPSHOTS = ROOT / 'data' / 'swebench' / 'snapshots'
MIRRORS = ROOT / 'data' / 'swebench' / 'git_mirrors'
OUT.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и воспроизводимый отбор 80 задач

Здесь создаются `issues.jsonl` (без patch) и отдельный `gold_labels.jsonl`. Не передавайте файл с метками функциям загрузки графа и retrieval.

In [ ]:
dataset = load_dataset('SWE-bench/SWE-bench_Verified', split='test')
print('Всего задач:', len(dataset), '| пример:', dataset[0]['instance_id'])
assert len(dataset) == 500, f'Ожидали 500 задач, получено {len(dataset)}'
counts = build_manifests(dataset, OUT, limit=80, dev_count=20, seed=42)
issues = read_jsonl(ISSUES)
print('Выборка:', counts, '| репозиториев:', len({r['repo'] for r in issues}))
pd.DataFrame(issues).groupby('split').agg(tasks=('instance_id','count'), repos=('repo','nunique'))

## 2. Снимки репозиториев на точном `base_commit`
Git worktree разделяет объектную базу репозитория между задачами и позволяет не клонировать одну и ту же библиотеку повторно. Ячейка возобновляемая: корректные существующие снимки пропускаются. Загрузка репозиториев может занять время и место на диске; `TASK_LIMIT` можно временно уменьшить для проверки. Перед итоговым измерением верните `None`.

In [ ]:
TASK_LIMIT = None  # например 3 для пробного запуска; None = все 80
SNAPSHOTS.mkdir(parents=True, exist_ok=True); MIRRORS.mkdir(parents=True, exist_ok=True)
def git(args, *, check=True):
    return subprocess.run(['git', *map(str,args)], text=True, capture_output=True, check=check)
def snapshot_for(row):
    repo_key = row['repo'].replace('/', '__')
    mirror = MIRRORS / (repo_key + '.git')
    snap = SNAPSHOTS / row['instance_id']
    if snap.exists():
        head = git(['-C', snap, 'rev-parse', 'HEAD']).stdout.strip()
        if head == row['base_commit']: return 'ready'
        raise RuntimeError(f'{snap} существует, но HEAD={head}; ожидается {row["base_commit"]}')
    if not mirror.exists():
        git(['clone', '--bare', '--filter=blob:none', f'https://github.com/{row["repo"]}.git', mirror])
    exists = git(['--git-dir', mirror, 'cat-file', '-e', row['base_commit'] + '^{commit}'], check=False)
    if exists.returncode: git(['--git-dir', mirror, 'fetch', '--filter=blob:none', 'origin', row['base_commit']])
    git(['--git-dir', mirror, 'worktree', 'add', '--detach', snap, row['base_commit']])
    head = git(['-C', snap, 'rev-parse', 'HEAD']).stdout.strip()
    if head != row['base_commit']: raise RuntimeError(f'Неверный commit для {row["instance_id"]}')
    return 'created'
todo = issues if TASK_LIMIT is None else issues[:TASK_LIMIT]
snapshot_status = []
for i, row in enumerate(todo, 1):
    snapshot_status.append(snapshot_for(row))
    if i % 10 == 0 or i == len(todo): print(f'{i}/{len(todo)} снимков')
print(pd.Series(snapshot_status).value_counts().to_dict())

In [ ]:
# Убедиться, что доступны все snapshots и каждый pinned commit совпадает.
snapshot_check = []
for row in issues:
    p = SNAPSHOTS / row['instance_id']
    if not p.is_dir(): continue
    head = git(['-C', p, 'rev-parse', 'HEAD']).stdout.strip()
    snapshot_check.append({'instance_id': row['instance_id'], 'expected': row['base_commit'], 'head': head, 'ok': head == row['base_commit']})
snapshot_check = pd.DataFrame(snapshot_check)
display(snapshot_check.groupby('ok').size().rename('snapshots'))
assert len(snapshot_check) == 80 and snapshot_check['ok'].all(), 'Подготовьте все 80 точных снимков перед экспериментом'

## 3. Проверка реальных графов
Парсер работает только с Python-файлами. Ошибки разбора и размер графа сохраняются для контроля покрытия. Эта ячейка строит графы один раз на задачу; retrieval затем использует тот же граф и одинаковые function chunks для всех методов.

In [ ]:
graph_cache = {}
graph_rows = []
for i, row in enumerate(issues, 1):
    repo_path = SNAPSHOTS / row['instance_id']
    t0 = time.perf_counter(); graph = scan_repository(repo_path); scan_sec = time.perf_counter() - t0
    chunks = make_chunks(repo_path, graph)
    graph_cache[row['instance_id']] = (graph, chunks)
    graph_rows.append({'instance_id': row['instance_id'], 'repo': row['repo'], 'scan_sec': scan_sec, 'chunks': len(chunks), **{k:v for k,v in graph.stats().items() if k != 'parse_errors'}, 'parse_error_files': len(graph.errors)})
    if i % 10 == 0 or i == len(issues): print(f'{i}/{len(issues)} графов')
graph_df = pd.DataFrame(graph_rows)
graph_df.to_csv(OUT / 'graph_inventory.csv', index=False)
display(graph_df[['scan_sec','chunks','nodes','edges','parse_error_files']].describe().round(2))

## 4. Основной прогон: BM25, Graph Expansion, PathPack
Latency retrieval измеряется отдельно от однократного AST-сканирования; `scan_sec` публикуется рядом. Все методы получают одни issue, graph, chunks и proxy-бюджеты. Результаты записываются без gold labels.

In [ ]:
METHODS = ('bm25', 'graph-expand', 'pathpack')
predictions = []
for i, row in enumerate(issues, 1):
    graph, chunks = graph_cache[row['instance_id']]
    for budget in BUDGETS:
        for method in METHODS:
            t0 = time.perf_counter()
            hits = select_context(chunks, graph, row['problem_statement'], method=method, budget=budget, seeds=3, hops=3)
            elapsed = time.perf_counter() - t0
            predictions.append({'instance_id': row['instance_id'], 'split': row['split'], 'method': method, 'budget': budget, 'latency_sec': elapsed, 'tokens_est': sum(h.chunk.tokens_est for h in hits), 'selected': [h.chunk.metadata() for h in hits]})
    if i % 10 == 0 or i == len(issues): print(f'{i}/{len(issues)} задач обработано')
with (OUT / 'predictions.jsonl').open('w', encoding='utf-8') as f:
    for row in predictions: f.write(json.dumps(row, ensure_ascii=False) + '\n')
print('Предсказаний:', len(predictions))

## 5. Изолированная оценка file recall и function recall
Gold patch применяется только здесь. File recall считает изменённые существовавшие Python-файлы. Function recall сопоставляет старые строки из diff hunk с функциями/методами AST на base commit; hunk без пересечения с функцией исключается из function recall и учитывается в `function_not_evaluable`. Для изменений на уровне модуля это приближение явно неполное.

In [ ]:
gold_rows = {r['instance_id']: r for r in read_jsonl(GOLD)}
source_rows = {r['instance_id']: r for r in dataset}
hunk_re = re.compile(r'^@@ -(\d+)(?:,(\d+))? \+\d+(?:,\d+)? @@')
def function_targets(patch, graph):
    targets = set(); current = None
    for line in patch.splitlines():
        if line.startswith('--- a/'):
            current = line[6:] if line[6:] != '/dev/null' else None
        elif line.startswith('--- /dev/null'):
            current = None
        elif line.startswith('@@') and current and current.endswith('.py'):
            m = hunk_re.match(line)
            if not m: continue
            start, count = int(m.group(1)), int(m.group(2) or 1)
            lo, hi = (start, start) if count == 0 else (start, start + count - 1)
            for node in graph.nodes.values():
                if node.path == current and node.kind in {'function','method'} and node.line_start <= hi and node.line_end >= lo:
                    targets.add(node.id)
    return targets
gold_functions = {}
for iid, graph_pair in graph_cache.items():
    gold_functions[iid] = function_targets(source_rows[iid]['patch'], graph_pair[0])
def recall_rows(predictions):
    scored=[]
    for p in predictions:
        labels = gold_rows[p['instance_id']]['gold_python_files']
        if not labels: continue
        selected_files = {x['path'] for x in p['selected']}
        file_hit = len(selected_files & set(labels)) / len(set(labels))
        targets = gold_functions[p['instance_id']]
        selected_ids = {x['id'] for x in p['selected']}
        fn_recall = len(selected_ids & targets) / len(targets) if targets else math.nan
        scored.append({**{k:p[k] for k in ('instance_id','split','method','budget','latency_sec','tokens_est')}, **{k:p[k] for k in ('graph_bonus','hops','redundancy_penalty') if k in p}, 'file_recall':file_hit, 'function_recall':fn_recall, 'function_targets':len(targets), 'useful_functions_per_1k_est_tokens':(len(selected_ids & targets) * 1000 / p['tokens_est']) if p['tokens_est'] else 0})
    return pd.DataFrame(scored)
scores = recall_rows(predictions)
summary = scores.groupby(['split','budget','method']).agg(tasks=('instance_id','nunique'), file_recall=('file_recall','mean'), function_recall=('function_recall','mean'), latency_sec=('latency_sec','mean'), tokens_est=('tokens_est','mean'), useful_functions_per_1k=('useful_functions_per_1k_est_tokens','mean')).reset_index()
summary.to_csv(OUT / 'main_results.csv', index=False)
display(summary.round(4))
print('File-recall coverage:', len({x['instance_id'] for x in issues if gold_rows[x['instance_id']]['gold_python_files']}), 'of', len(issues), 'tasks; without gold Python files:', sum(not gold_rows[x['instance_id']]['gold_python_files'] for x in issues))
print('Function recall evaluated:', int(scores['function_recall'].notna().sum()), '| excluded:', int(scores['function_recall'].isna().sum()))

## 6. Абляции PathPack на dev
Сравниваем влияние графового бонуса, длины обхода и штрафа за лексическое сходство уже выбранных фрагментов. Тестовые задачи в подборе параметров не участвуют. Нулевой штраф воспроизводит текущий PathPack.

In [ ]:
ABLATION = [(g,h,r) for g in (0.0, 0.75, 1.5) for h in (1,2,3) for r in (0.0,0.25,0.5)]
dev_rows = [x for x in issues if x['split'] == 'dev']
ablation_predictions=[]
for row in dev_rows:
    graph, chunks = graph_cache[row['instance_id']]
    for budget in BUDGETS:
        for graph_bonus, hops, redundancy_penalty in ABLATION:
            t0=time.perf_counter()
            hits=select_context(chunks, graph, row['problem_statement'], method='pathpack', budget=budget, seeds=3, hops=hops, graph_bonus=graph_bonus, redundancy_penalty=redundancy_penalty)
            ablation_predictions.append({'instance_id':row['instance_id'],'split':'dev','method':'pathpack','budget':budget,'graph_bonus':graph_bonus,'hops':hops,'redundancy_penalty':redundancy_penalty,'latency_sec':time.perf_counter()-t0,'tokens_est':sum(h.chunk.tokens_est for h in hits),'selected':[h.chunk.metadata() for h in hits]})
ablation_scores=recall_rows(ablation_predictions)
ablation_summary=ablation_scores.groupby(['budget','graph_bonus','hops','redundancy_penalty']).agg(tasks=('instance_id','nunique'),file_recall=('file_recall','mean'),function_recall=('function_recall','mean'),latency_sec=('latency_sec','mean'),tokens_est=('tokens_est','mean')).reset_index()
ablation_summary.to_csv(OUT / 'dev_ablations.csv',index=False)
display(ablation_summary[ablation_summary['budget']==2048].sort_values(['file_recall','function_recall'],ascending=False).head(15).round(4))

## 7. Замораживаем выбор на dev и повторно считаем held-out test
Правило выбора ниже заранее задано: лучший macro file recall на dev при бюджете 2048; равенства разрешаются меньшим штрафом, затем меньшим числом hops и бонусом. После этого выбранная конфигурация применяется к test. Сохраните таблицы и выбранные параметры вместе с отчетом.

In [ ]:
dev2048=ablation_summary[ablation_summary['budget']==2048].sort_values(['file_recall','redundancy_penalty','hops','graph_bonus'],ascending=[False,True,True,True])
best=dev2048.iloc[0]
frozen={'graph_bonus':float(best.graph_bonus),'hops':int(best.hops),'redundancy_penalty':float(best.redundancy_penalty),'selection_rule':'max dev file recall @2048; tie: lower redundancy, fewer hops, lower graph bonus'}
print('Замороженные параметры:', frozen)
final_test=[]
for row in [x for x in issues if x['split']=='test']:
    graph,chunks=graph_cache[row['instance_id']]
    for budget in BUDGETS:
        t0=time.perf_counter()
        hits=select_context(chunks,graph,row['problem_statement'],method='pathpack',budget=budget,seeds=3,**{k:frozen[k] for k in ('graph_bonus','hops','redundancy_penalty')})
        final_test.append({'instance_id':row['instance_id'],'split':'test','method':'pathpack-dev-tuned','budget':budget,'latency_sec':time.perf_counter()-t0,'tokens_est':sum(h.chunk.tokens_est for h in hits),'selected':[h.chunk.metadata() for h in hits]})
final_scores=recall_rows(final_test)
final_summary=final_scores.groupby(['budget','method']).agg(tasks=('instance_id','nunique'),file_recall=('file_recall','mean'),function_recall=('function_recall','mean'),latency_sec=('latency_sec','mean'),tokens_est=('tokens_est','mean'),useful_functions_per_1k=('useful_functions_per_1k_est_tokens','mean')).reset_index()
final_summary.to_csv(OUT / 'heldout_pathpack.csv',index=False)
(OUT / 'frozen_config.json').write_text(json.dumps(frozen,indent=2)+'\n',encoding='utf-8')
display(final_summary.round(4))

## Интерпретация и отчет
Ключевое сравнение — PathPack против Graph Expansion на одинаковых 60 held-out задачах. Публикуйте абсолютную разницу и доверительный интервал по парному bootstrap на уровне задачи; отдельно покажите долю задач без gold Python-файлов и function targets. Сравнение статистически неубедительно при малом числе оцениваемых задач.

Отчет этого блокнота сохраняется в `output/benchmark/`: манифесты, инвентарь графов, сырые retrieval-предсказания, сводная таблица, dev-абляции и замороженная конфигурация. Не объединяйте gold patch с issue manifest и не используйте test для выбора настроек.

In [ ]:
import random
def paired_bootstrap(split, budget, metric='file_recall', n=5000, seed=42):
    sub=scores[(scores.split==split) & (scores.budget==budget)]
    wide=sub.pivot(index='instance_id',columns='method',values=metric).dropna()
    rng=random.Random(seed); pairs=[(a,b) for a,b in zip(wide['pathpack'],wide['graph-expand'])]
    if not pairs: return {'split':split,'budget':budget,'metric':metric,'n':0,'delta':None,'ci95':None}
    diffs=[a-b for a,b in pairs]
    draws=[sum(rng.choice(diffs) for _ in diffs)/len(diffs) for _ in range(n)]
    draws.sort(); mean=sum(diffs)/len(diffs)
    return {'split':split,'budget':budget,'metric':metric,'n':len(diffs),'pathpack_minus_graph_expand':mean,'ci95_low':draws[int(.025*n)],'ci95_high':draws[min(n-1,int(.975*n))]}
ci_table=pd.DataFrame([paired_bootstrap('test',b,m) for b in BUDGETS for m in ('file_recall','function_recall')])
ci_table.to_csv(OUT / 'paired_bootstrap_test.csv',index=False)
display(ci_table.round(4))